In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MokhtarrSamir/FlyRank-internship-ML.git"
REPO_DIR = "FlyRank-internship-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

Working dir: /content/FlyRank-internship-ML


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MokhtarrSamir/FlyRank-internship-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This lane frames a classification task whose output feeds a ranking. The model first predicts is_declining_label (1 = page's trend is "down", 0 = otherwise) as a binary classification problem. But we don't stop at the 0/1 prediction — we take the model's predicted probability for each page, because probability is a continuous score that lets us rank pages precisely, whereas the raw binary label would leave many pages tied at the same value. Sorting pages by that probability produces the final refresh-priority queue.

It can't be clustering, because we aren't grouping pages into unlabeled segments — every page has an observed outcome (trend_direction) we're trying to predict. It also isn't pure signal analysis / EDA, because EDA only describes correlations without committing to a specific predictive target; here we commit to is_declining_label as the thing to predict.

In [ ]:
# This section is conceptual — no computation needed.
# The task-type reasoning is written in the markdown cell above.

## 2. Target or proxy

Target: is_declining_label
The target for this lane is is_declining_label, a binary column derived directly from trend_direction == "down" (1 = declining, 0 = otherwise). trend_direction itself is computed by comparing impressions_last_30d to impressions_prev_30d: a drop of more than 20% is labeled down, a rise of more than 20% is up, and anything in between is stable (with new/flat covering the zero-impression edge cases).

This is a proxy label, not an observed future outcome. The bucket is computed entirely from the current 90-day window — there is no decision point after which we measure a real future result (e.g., "did the page recover after a refresh?"). It's a snapshot classification, not a forward-looking outcome. It's also worth noting that the 0 class is not a clean "opposite" of declining — it lumps together four different situations (up, flat, stable, new) into one bucket, which makes the negative class noisier than it looks.

Leakage note: trend_direction and trend_pct are the source of this label, so they can never appear as model features later — using them would mean the model is just reading the answer back out of the input.

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# show trend_direction distribution
print(df["trend_direction"].value_counts())
print()

# recompute the label the same way the pipeline does
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*Metric: Precision@50, reported next to the base rate
I'll use Precision@50 as the primary success metric: of the top 50 pages the model ranks first for review, what fraction are actually labeled declining? This fits the ranking framing directly — the reviewer has limited time and can't check every page, so what matters is the quality of the top of the list, not overall accuracy across all 30,000 rows. A model can have mediocre overall accuracy and still be extremely useful if its top-50 recommendations are consistently right.

I chose K=50 to match the reference pipeline (03_train_model.py selects its best model by precision_at_50), which also lets me compare my own numbers directly against the shipped baseline (outputs/model_results.json: baseline rule = 0.240, random forest = 0.740).

Precision@50 is only meaningful next to the base rate — the declining rate across the full dataset (~54%, computed in Section 2). Without that reference point, a Precision@50 of e.g. 0.60 could just mean the model is barely better than picking 50 random pages. The real signal is the gap between Precision@50 and the base rate.*

In [4]:
print("Base rate (declining):", df["is_declining_label"].mean())

Base rate (declining): 0.5420666666666667


## 4. The unit of analysis, as a real dataframe

One row in this dataset represents one content page (content_id), described by trailing 90-day search and engagement metrics (impressions, clicks, average position, CTR, etc.), tied to one client (client_id). The dataset has 30,000 such rows across 32 clients.

In [5]:
# The unit of analysis: one row = one content page (content_id),
# described by trailing 90-day search/engagement metrics.
sample_cols = [
    "content_id", "client_id", "impressions_90d", "clicks_90d",
    "avg_position", "ctr", "content_age_days", "days_since_last_update",
    "trend_direction", "is_declining_label"
]
df[sample_cols].head(10)

,content_id,client_id,impressions_90d,clicks_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,0.76,187,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,0.05,445,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,141,20,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,0.49,463,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,0.13,263,14,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,8.5,0.03,147,20,down,1
6,content_9a34b442b552,client_8722616204,20,0,7.0,0.00,90,20,down,1
7,content_a63219c6e95a,client_19581e27de,1724,1,21.2,0.06,445,22,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,46.0,0.09,90,20,down,1
9,content_c27558df2b0c,client_19581e27de,1240,2,4.9,0.16,257,104,down,1


## 5. Why ML beats a fixed rule here

*`02_baseline_score.py` implements a hand-written rule, and `03_train_model.py` trains a random forest on the same label. Results from `outputs/model_report.md`:

| Method | Precision@50 |
|---|---:|
| baseline rule | 0.240 |
| random forest | 0.740 |

The rule's Precision@50 (0.240) is actually **below the dataset's base rate (~0.54)** — worse than random guessing. This happens because the rule treats conditions as independent (e.g. "stale AND visible"), while signals like `avg_position`, `impressions_90d`, and `content_age_days` interact non-linearly — the effect of one depends on the value of another, which a fixed if/else can't capture without endless manual thresholds. A learned model finds these interactions directly from the data instead.

This also rules out clustering or pure EDA: a beatable rule already exists with a measurable metric, which is exactly where a learned model earns its complexity.*

In [6]:
# This section is conceptual — the evidence (baseline vs model Precision@50)
# comes from the already-committed outputs/model_report.md, no new computation needed here.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.